# Dataset : Ablated : 'JR_22-11-17'

## Imports

In [ ]:
from lineagetree import read_from_mastodon
import numpy as np
import matplotlib.pyplot as plt
import h5py
from skimage.segmentation import watershed
import h5py
import traceback
import subprocess
import sys
import zlib
import warnings
import hdf5plugin

# CUSTOM FUNCTIONS
import watermasks.utils as utils
import watermasks.main as main

# TO RELOAD THEM FOR TESTING
import importlib

# --- IMPORT DATA AND FILES ---
path_h5, path_xml, path_mast, dir_input_images, dir_output_masks = utils.paths('JR_22-11-17_deconvolved')

In [ ]:
# IMPORT REG. MATRICES AND LINEAGETREE
R_of_t = utils.import_registration_mats_multiview(path_xml, 0, 500)
lT = read_from_mastodon(path_mast)

## Opening a working timepoint

In [ ]:
TP = 300 # THIS TIMEPOINT WORKS (ONE OF THE FEW)
VIEW='2' # The view with the deconvolved dataset

# IMPORT INPUT IMAGE
with h5py.File(path_h5, "r") as f5:
    raw_image = f5[f't00{TP}'][f's0{VIEW}']['2']['cells'][:]

In [ ]:
# CALCULATE SCALING FACTORS
# shape_full_h5 = np.array([300, 1888, 2100])
shape_full_h5 = [1063, 1145, 1142] # as measured
shape_raw_image = np.array(np.shape(raw_image))
scaling = shape_raw_image/shape_full_h5
print(f'full image shape = {shape_full_h5} ---- 2nd h5 layer has shape = {shape_raw_image} -----  scaling ratio is [z, y, x] = {scaling}')

In [ ]:
# GET THE REGISTERED MASTODON POSITIONS AT CURRENT TIMEPOIN
reg_pos_at_t = []
for mastodon_id_t in lT.time_nodes[TP]:
    z, y, x = utils.registered_position_of_id_in_t(mastodon_id_t, lT, TP, R_of_t, '2', scaling, Transformations_In_Reverse=True)
    reg_pos_at_t.append([x, y, z])

im_for_ws = main.preprocess_im(raw_image, erosion_radius=5)

seeds_pos, seeds_array = main.get_seeds(lT, TP, VIEW, R_of_t, scaling, raw_image)

ws, all_seeds, missed_seeds, stat_table = main.ws_adaptive_mask(im_for_ws, seeds_array, min_int=np.min(im_for_ws), max_int=np.max(im_for_ws),
                                                                percentage_list=[100, 99.99, 99.9, 99, 97, 95, 80, 85, 70, 75, 60, 50,40, 30, 20, 10], min_vol = 10, max_vol=1000)

In [ ]:
from beautifultable import BeautifulTable

table = BeautifulTable()
table.columns.header = ['Percentage', 'Percentile Value', 'Found Seeds', 'Missed Seeds']
for i in range(len(stat_table)):
    table.rows.insert(i, stat_table[i])
print(table)
print(f'Out of {len(all_seeds)} anotations in total, missed seeds: {stat_table[-1][-1]}({stat_table[-1][-1]/len(all_seeds)*100:.3g}%)')

In [ ]:
import napari

viewer = napari.Viewer()
viewer.add_image(im_for_ws)
viewer.add_labels(ws)
viewer.add_points(reg_pos_at_t, size = 1, face_color='#aaff00ff', border_color='black')
napari.run()

## Working with pratially corrupted h5 files

In [ ]:
# # COPY THE CORRUPTED DATASET TO A SEPERATE FILE

# with h5py.File(path_h5, "r") as f_src, h5py.File("corrupt.h5", "w") as f_dst:
#     src_dset = f_src['t00402']['s02']['2']['cells']

#     # Create dataset in new file (NO compression = safest)
#     dst_dset = f_dst.create_dataset(
#         "cells",
#         data=src_dset[:],          # deep copy into RAM, then write
#         dtype=src_dset.dtype,
#         chunks=src_dset.chunks
#     )

#     # Copy attributes (metadata)
#     for k, v in src_dset.attrs.items():
#         print(v)
#         dst_dset.attrs[k] = v

## Count the number of corrupted chunks in the dataseet by timepoint

In [ ]:
timepoints = range(0,501)
num_corrupted_chunks = []

for tmp in timepoints:
    with h5py.File(path_h5, "r") as f5:
        test_image_decon = f5[f't{tmp:05}'][f's0{VIEW}']['2']['cells'] # this is the group

        bad_chunks = []

        im_shape = test_image_decon.shape
        cz, cy, cx = test_image_decon.chunks
        datatype = test_image_decon.dtype

    for z in range(0, im_shape[0], cz):
        for y in range(0, im_shape[1], cy):
            for x in range(0, im_shape[2], cx):
                try:
                    _ = test_image_decon[z:z+cz, y:y+cy, x:x+cx]
                except Exception:
                    bad_chunks.append((z, y, x))

    num_corrupted_chunks.append(len(bad_chunks))

In [ ]:
num_corrupted_chunks

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
# --- PLOT ---
plt.plot(timepoints, num_corrupted_chunks)
plt.xlabel("Timepoint", size=12)
plt.ylabel("# of inaccessible chunks", size=12)
plt.title("Corrupted Chunks of JR_22-11-17:deconvolved", weight='bold', size=12)
plt.tick_params(size=7)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.show()

## Recreate the image, by replacing the corrupted chunks with black boxes

In [ ]:
# --- RECREATE THE IMAGE BY REPLAING THE CORRUPTED CHUNKS WITH BLACK BOXES ---
out = np.zeros(im_shape, dtype=datatype)

for z in range(0, im_shape[0], cz):
    for y in range(0, im_shape[1], cy):
        for x in range(0, im_shape[2], cx):
            try:
                out[z:z+cz, y:y+cy, x:x+cx] = \
                    test_image_decon[z:z+cz, y:y+cy, x:x+cx]
            except Exception:
                # corrupted chunk → leave zeros
                pass

In [ ]:
viewer = napari.Viewer()
# viewer.add_image(test_image_decon)
viewer.add_image(out)
napari.run()

## Continue with image proccessing and mask creation

In [ ]:
# LOAD REG. MATRICES AND LINEAGETREE
R_of_t = import_registration_mats_multiview(path_xml, TP-5,TP+5)
lT = read_from_mastodon(path_mast)

shape_test_image = np.array(np.shape(out))
# shape_full_h5 = np.asarray([304, 1602, 2140]) # This is the h5 with the original images : view 0 and view 1
shape_full_deconvolved = np.asarray([1063, 1145, 1142]) # This has been checked by loading the 0-th decomression h5 level of the deconvolved file

# scaling = scaling * np.shape(test_image)/shape_test_image # this had to be done because I couldn't open the first (and second) layer of the h5  for the devonvolved dataset
scaling = shape_test_image/shape_full_deconvolved # this had to be done because I couldn't open the first (and second) layer of the h5  for the devonvolved dataset
# print(f'full image shape = {shape_full_deconvolved}\n2nd h5 layer has shape = {shape_test_image} \n-scaling ratio is [z, y, x] = {scaling}')

# GET THE REGISTERED MASTODON POSITIONS
reg_pos_at_t = []
for mastodon_id_t in lT.time_nodes[TP]:
    x, y, z = registered_position_of_id_in_t(mastodon_id_t, lT, TP, R_of_t, VIEW, scaling, Transformations_In_Reverse=False)
    reg_pos_at_t.append([x, y, z])
reg_pos_at_t = np.asarray(reg_pos_at_t)

# --- VISUALIZE THE ANNOTATIONS ---
an_space = paint_annotations(TP, lT, np.array(shape_test_image), R_of_t, scaling, VIEW, 1, Transformations_In_Reverse=False)

In [ ]:
plt.hist(out.ravel(), bins=100)
plt.yscale('log')
plt.xscale('log')

## Testing if the problem arises form the filter sequence

In [ ]:
# OPEN A PROBLEMATIC TIMEPOINT
TP = 300 # THIS TIMEPOINT DOES NOT WORK
VIEW='2' # The view with the deconvolved dataset

# # IMPORT INPUT IMAGE
# with h5py.File(path_h5, "r") as f5:
#     raw_image = f5[f't00{TP}'][f's0{VIEW}']['2']['cells'][:]

import h5py.h5z as h5z

# Check filter info at a lower level
with h5py.File(path_h5, "r") as f5:
    dataset = f5[f't00{TP}'][f's0{VIEW}']['2']['cells']
    
    # Get filter configuration
    dcpl = dataset.id.get_create_plist()
    
    for i in range(dcpl.get_nfilters()):
        info = dcpl.get_filter(i)
        print(f"\nFilter {i}:")
        print(f"  ID: {info[0]}")
        print(f"  Flags: {info[1]}")  # Check mandatory/optional flags
        print(f"  CD values: {info[2]}")
        print(f"  Name: {info[3]}")
        
        # Check if filter is marked as optional (can fail without error)
        if info[1] & h5z.FLAG_OPTIONAL:
            print("  -> Filter is OPTIONAL")
        if info[1] & h5z.FLAG_MANDATORY:
            print("  -> Filter is MANDATORY")

In [ ]:
with h5py.File(path_h5, "r") as f5:
    # dataset = f5[f't00{TP}'][f's0{VIEW}']['2']['cells']
    
    # # Try to read a failing chunk
    # failing_t = YOUR_FAILING_TIMEPOINT
    
    try:
        data = f5[f't00{TP}'][f's0{VIEW}']['2']['cells'][:]
    except OSError as e:
        error_msg = str(e)
        print(f"Error message: {error_msg}")
        
        # Check if error mentions specific filter
        if 'deflate' in error_msg.lower() or 'gzip' in error_msg.lower():
            print("→ Deflate/gzip is likely failing")
        elif 'scaleoffset' in error_msg.lower():
            print("→ Scaleoffset is likely failing")
        else:
            print("→ Generic filter failure (unclear which)")

In [ ]:
import h5py
import traceback

with h5py.File(path_h5, "r") as f5:
    try:
        data = f5[f't00{TP}'][f's0{VIEW}']['2']['cells'][:]
    except OSError as e:
        # Get full traceback
        print("Full error traceback:")
        traceback.print_exc()
        print("\nError details:")
        print(f"  Exception type: {type(e)}")
        print(f"  Exception args: {e.args}")
        
        # Try to get HDF5 error stack
        try:
            import h5py._errors
            h5py._errors.silence_errors()  # This might reveal more
        except:
            pass

### Slow reading

In [ ]:
with h5py.File(path_h5, "r") as f5:
    dataset = f5[f't00{TP}'][f's0{VIEW}']['2']['cells']
    shape = dataset.shape
    print(shape)
    data_slow =np.zeros(shape, dtype=np.int16)

    try:
        for i in range(shape[0]):
            data_slow[i] = dataset[0:shape[1]:1, 0:shape[2]:1][0]
            print("Slow reader worked!")
            print(f"Data shape: {data_slow.shape}")
            print(f"Data range: {data_slow.min()} to {data_slow.max()}")
    except OSError as e:
        print(f"Slow reader also failed: {e}")

In [ ]:
with h5py.File(path_h5, "r") as f5:
    dataset = f5[f't00{TP}'][f's0{VIEW}']['2']['cells']
    shape = dataset.shape
    data_slow =np.zeros(shape, dtype=np.int16)

    for k in range(0, shape[0]):
        try:
            dataset[[k]][0]
            print(f"{k} : Slow reader worked!")
            # print(f"Data shape: {data_slow.shape}")
            # print(f"Data range: {data_slow.min()} to {data_slow.max()}")
        except OSError as e:
            print(f"{k} : Slow reader failed: {e}")

In [ ]:
import h5py

# Monkey-patch to disable fast reader
original_getitem = h5py.Dataset.__getitem__

def slow_getitem(self, args):
    # Force the slow path by using the parent's method
    # but with modified selection
    if isinstance(args, (int, slice)):
        args = (args,)
    
    # Convert to explicit selection that bypasses fast reader
    if isinstance(args, tuple) and len(args) > 0:
        if isinstance(args[0], int):
            # Convert int to list to force slow path
            new_args = ([args[0]],) + args[1:]
            result = original_getitem(self, new_args)
            return result[0]
    
    return original_getitem(self, args)

# Apply the patch
h5py.Dataset.__getitem__ = slow_getitem

# # Now try reading
# with h5py.File(path_h5, "r") as f5:
#     dataset = f5[f't00{TP}'][f's0{VIEW}']['2']['cells']
    
#     try:
#         data = dataset[:]
#         print("Success with patched reader!")
#     except OSError as e:
#         print(f"Still failed: {e}")

In [ ]:
with h5py.File(path_h5, "r") as f5:
    dataset = f5[f't00{TP}'][f's0{VIEW}']['2']['cells']
    
    # Use low-level HDF5 API
    # Create output array
    output_shape = dataset.shape[1:]  # Shape without z dimension
    out = np.empty(output_shape, dtype=dataset.dtype)
    
    undreadable_z_slice = 150
    # Create memory and file spaces
    mspace = h5py.h5s.create_simple(output_shape)
    fspace = dataset.id.get_space()
    
    # Select the hyperslab in file
    fspace.select_hyperslab((undreadable_z_slice, 0, 0), (1, *output_shape))
    
    try:
        # Read directly using low-level API
        dataset.id.read(mspace, fspace, out) # this is what really reads the data
        print("Success with low-level API!")
        print(f"Data shape: {out.shape}")
    except Exception as e:
        print(f"Low-level read failed: {e}")

In [ ]:
import napari

viewer = napari.Viewer()
viewer.add_image(out)
napari.run()

In [ ]:
with h5py.File(path_h5, "r") as f5:
    dataset = f5[f't00{TP}'][f's0{VIEW}']['2']['cells']
    
    # Assuming dataset shape is (z, y, x)
    print(f"Dataset shape: {dataset.shape}")
    
    num_z = dataset.shape[0]
    output_shape_2d = dataset.shape[1:]  # (y, x), exclude z
    
    # Pre-allocate full 3D array
    all_slices = np.empty(dataset.shape, dtype=dataset.dtype)
    
    # Create memory space for one slice
    mspace = h5py.h5s.create_simple(output_shape_2d)
    fspace = dataset.id.get_space()
    
    # Read each z-slice
    for z in range(num_z):
        # Select this z-slice from file
        fspace.select_hyperslab((z, 0, 0), (1, *output_shape_2d))
        
        # Read directly into the pre-allocated array
        dataset.id.read(mspace, fspace, all_slices[z])
        
        if z % 10 == 0:
            print(f"Read z-slice {z}/{num_z}")
    
    print(f"Final array shape: {all_slices.shape}")

In [ ]:
import napari

viewer = napari.Viewer()
viewer.add_image(data)
napari.run()

## Opening a reasaved timepoint

In [ ]:
path_h5, path_xml, path_mast, dir_input_images, dir_output_masks = utils.paths('JR_22-11-17_deconvolved_test')

In [ ]:
with h5py.File(path_h5, "r") as f5:
    image = f5['t2']['channel0'][:]

In [ ]:
shape_full_h5 = [1063, 1145, 1142] # as measured
shape_raw_image = np.array(np.shape(image))
scaling = shape_raw_image/shape_full_h5
print(f'full image shape = {shape_full_h5} ---- 2nd h5 layer has shape = {shape_raw_image} -----  scaling ratio is [z, y, x] = {scaling}')

In [ ]:
# GET THE REGISTERED MASTODON POSITIONS AT CURRENT TIMEPOINT
importlib.reload(main)

reg_pos_at_t = []
for mastodon_id_t in lT.time_nodes[TP]:
    z, y, x = utils.registered_position_of_id_in_t(mastodon_id_t, lT, TP, R_of_t, '2', scaling, Transformations_In_Reverse=True)
    reg_pos_at_t.append([x, y, z])

im_for_ws = image
# im_for_ws = main.preprocess_im(image, erosion_radius=7)

seeds_pos, seeds_array = main.get_seeds(lT, TP, VIEW, R_of_t, scaling, image)

ws, all_seeds, missed_seeds, stat_table, vols_dict = main.ws_adaptive_mask(im_for_ws, seeds_array, min_int=np.percentile(im_for_ws.ravel(), 80), max_int=np.max(image),
                                                                percentage_list=[100, 99.99, 99.9, 99, 97, 95, 80], min_vol = 10, max_vol= 500, return_vols_dict=True)

In [ ]:
from beautifultable import BeautifulTable

table = BeautifulTable()
table.columns.header = ['Percentage', 'Percentile Value', 'Found Seeds', 'Missed Seeds']
for i in range(len(stat_table)):
    table.rows.insert(i, stat_table[i])
print(table)
print(f'Out of {len(all_seeds)} anotations in total, missed seeds: {stat_table[-1][-1]}({stat_table[-1][-1]/len(all_seeds)*100:.3g}%)')

In [ ]:
import napari

viewer = napari.Viewer()
viewer.add_image(image)
# viewer.add_image(im_for_ws)
# viewer.add_labels(ws)
# viewer.add_points(reg_pos_at_t, size = 5, face_color='#aaff00ff', border_color='black')
napari.run()

In [ ]:
np.count_nonzero(ws == 261)

## Copy the dataset using the h5py library

In [ ]:
import h5py

TP = 400
# Open source file
with h5py.File(path_h5, 'r') as src:
    # Create new file for specific timepoint
    with h5py.File('filter_error.h5', 'w') as dst:
        # Copy specific timepoint (adjust path to match your structure)
        src.copy(rf't00{TP}/s02/2/cells', dst)  # or whatever your timepoint path is

# with h5py.File('filter_error.h5', 'r') as testh5:
#     test = testh5['cells'][:]

In [ ]:
with h5py.File('filter_error.h5', 'r') as testh5:
    test = testh5['cells'][:]

## Load resaved datasets that are created using Multiview Reconstruction > Reasave As >

Firts try the one that uses the default Fiji settings

In [ ]:
TP = 420

path_h5, path_xml, path_mast, dir_input_images, dir_output_masks = utils.paths('JR_22-11-17_400_fix_default')
with h5py.File(path_h5, 'r') as f5:
    # for i in f5:
    #     print(i)
    test_default = f5['t00420']['s02']['1']['cells'][:] # -> doen't work (if it has a deflation filter)

In [ ]:
np.shape(test_default)

In [ ]:
import napari

viewer = napari.Viewer()
viewer.add_image(test_default)
# viewer.add_image(im_for_ws)
# viewer.add_labels(ws)
# viewer.add_points(reg_pos_at_t, size = 5, face_color='#aaff00ff', border_color='black')
napari.run()

In [ ]:
TP = 420

path_h5, path_xml, path_mast, dir_input_images, dir_output_masks = utils.paths('JR_22-11-17_400_fix_manual')

with h5py.File(path_h5, 'r') as f5:
    dset = f5['t00420']['s02']['0']['cells']
    print(dset.shape)
    # test_manual = f5['t00420']['s02']['2']['cells'][:]

# Runing watershed on the deconvolved dataset

### where all views, illuminations and dowsampling levels are inside

In [ ]:
from lineagetree import read_from_mastodon
import numpy as np
import matplotlib.pyplot as plt
import h5py
from skimage.segmentation import watershed
import h5py
import traceback
import subprocess
import sys
import zlib
import warnings
import hdf5plugin

# CUSTOM FUNCTIONS
import watermasks.utils as utils
import watermasks.main as main

# TO RELOAD THEM FOR TESTING
import importlib

In [ ]:
TP = 420 # THIS TIMEPOINT WORKS (ONE OF THE FEW)
VIEW='2' # The view with the deconvolved dataset

path_h5, path_xml, path_mast, dir_input_images, dir_output_masks = utils.paths('JR_22-11-17_400_fix_default')
with h5py.File(path_h5, 'r') as f5:
    # for i in f5['t00420']['s01']['0']:
    #     print(i)
    raw_image = f5['t00420']['s02']['1']['cells'][:] # -> doen't work (if it has a deflation filter)
    # raw_image = f5['t00420']['s00']['0']['cells'] # to measure the original image size
    # print(raw_image.shape)

# IMPORT REG. MATRICES AND LINEAGETREE


In [ ]:
R_of_t = utils.import_registration_mats_multiview(path_xml, 0, 500)
lT = read_from_mastodon(path_mast)

the file that is read, 400_fix_default.h5 is 9 GB!!

In [ ]:
# CALCULATE SCALING FACTORS
# shape_full_h5 = np.array([300, 1888, 2100])
shape_full_h5 = [1063, 1145, 1142] # as measured
shape_raw_image = np.array(np.shape(raw_image))
scaling = shape_raw_image/shape_full_h5
print(f'full image shape = {shape_full_h5} ---- 2nd h5 layer has shape = {shape_raw_image} -----  scaling ratio is [z, y, x] = {scaling}')

In [ ]:
# GET THE REGISTERED MASTODON POSITIONS AT CURRENT TIMEPOIN
reg_pos_at_t = []
for mastodon_id_t in lT.time_nodes[TP]:
    z, y, x = utils.registered_position_of_id_in_t(mastodon_id_t, lT, TP, R_of_t, '2', scaling, Transformations_In_Reverse=True)
    reg_pos_at_t.append([x, y, z])

# im_for_ws = main.preprocess_im(raw_image, erosion_radius=5)
im_for_ws = raw_image # you might want to use the deconvlolved image as is

seeds_pos, seeds_array = main.get_seeds(lT, TP, VIEW, R_of_t, scaling, raw_image)

ws, all_seeds, missed_seeds, stat_table = main.ws_adaptive_mask(im_for_ws, seeds_array, min_int=np.min(im_for_ws), max_int=np.max(im_for_ws),
                                                                percentage_list=[100, 99.99, 99.9, 99, 97, 95, 80, 85, 70, 75, 60, 50,40, 30, 20, 10], min_vol = 10, max_vol=1000)

In [ ]:
from beautifultable import BeautifulTable

table = BeautifulTable()
table.columns.header = ['Percentage', 'Percentile Value', 'Found Seeds', 'Missed Seeds']
for i in range(len(stat_table)):
    table.rows.insert(i, stat_table[i])
print(table)
print(f'Out of {len(all_seeds)} anotations in total, missed seeds: {stat_table[-1][-1]}({stat_table[-1][-1]/len(all_seeds)*100:.3g}%)')

In [ ]:
import napari

viewer = napari.Viewer()
viewer.add_image(im_for_ws)
viewer.add_labels(ws)
viewer.add_points(reg_pos_at_t, size = 1, face_color='#aaff00ff', border_color='black')
napari.run()

### where all views, illuminations but only level 1 of downsampling is inside

In [ ]:
from lineagetree import read_from_mastodon
import numpy as np
import matplotlib.pyplot as plt
import h5py
from skimage.segmentation import watershed
import h5py

# CUSTOM FUNCTIONS
import watermasks.utils as utils
import watermasks.main as main

# TO RELOAD THEM FOR TESTING
import importlib

In [ ]:
TP = 420 # THIS TIMEPOINT WORKS (ONE OF THE FEW)
VIEW='2' # The view with the deconvolved dataset

path_h5, path_xml, path_mast, dir_input_images, dir_output_masks = utils.paths('JR_22-11-17_400_fix_manual')
with h5py.File(path_h5, 'r') as f5:
    # for i in f5['t00420']['s02']['0']:
    #     print(i)
    # raw_image = f5['t00420']['s02']['1']['cells'][:] # -> doen't work (if it has a deflation filter)
    # raw_image = f5['t00420']
    raw_image = f5['t00420']['s02']['0']['cells'][:]
print(raw_image.shape)

In [ ]:
# IMPORT REG. MATRICES AND LINEAGETREE
R_of_t = utils.import_registration_mats_multiview(path_xml, 0, 500)
lT = read_from_mastodon(path_mast)

In [ ]:
shape_full_h5 = np.array([300, 1888, 2100]) # as measured in T7_3 data (dir = H:)
shape_raw_image = np.array(np.shape(raw_image))
scaling = shape_raw_image/shape_full_h5
print(f'full image shape = {shape_full_h5}\n2nd h5 layer has shape = {shape_raw_image} \n-scaling ratio is [z, y, x] = {scaling}')

# GET THE REGISTERED MASTODON POSITIONS AT CURRENT TIMEPOIN
seeds_pos, seeds_array = main.get_seeds(lT, TP, VIEW, R_of_t, scaling, raw_image)

# PREPROCCESS IMAGE TO CREATE WATERSHED INPUT
im_for_ws = main.preprocess_im(raw_image)

In [ ]:
from beautifultable import BeautifulTable

table = BeautifulTable()
table.columns.header = ['Percentage', 'Percentile Value', 'Found Seeds', 'Missed Seeds']
for i in range(len(stat_table)):
    table.rows.insert(i, stat_table[i])
print(table)
print(f'Out of {len(all_seeds)} anotations in total, missed seeds: {stat_table[-1][-1]}({stat_table[-1][-1]/len(all_seeds)*100:.3g}%)')

In [ ]:
import napari

viewer = napari.Viewer()
viewer.add_image(im_for_ws)
viewer.add_labels(ws)
viewer.add_points(reg_pos_at_t, size = 5, face_color='#aaff00ff', border_color='black')
napari.run()

## Compairing results between deconvolved and non-deconvolved dataset

In [ ]:
from lineagetree import read_from_mastodon
import numpy as np
import matplotlib.pyplot as plt
import h5py
from skimage.segmentation import watershed
import h5py

# CUSTOM FUNCTIONS
import watermasks.utils as utils
import watermasks.main as main

# TO RELOAD THEM FOR TESTING
import importlib

In [ ]:
TP = 420 # THIS TIMEPOINT WORKS (ONE OF THE FEW)
VIEW='0' # The view with the deconvolved dataset

path_h5, path_xml, path_mast, dir_input_images, dir_output_masks = utils.paths('JR_22-11-17_400_fix_manual')
with h5py.File(path_h5, 'r') as f5:
    # for i in f5['t00420']['s02']['0']:
    #     print(i)
    # raw_image = f5['t00420']['s02']['1']['cells'][:] # -> doen't work (if it has a deflation filter)
    # raw_image = f5['t00420']
    raw_image = f5[f't00{TP}'][f's0{VIEW}']['0']['cells'][:]

In [ ]:
# IMPORT REG. MATRICES AND LINEAGETREE
R_of_t = utils.import_registration_mats_multiview(path_xml, 0, 500)
lT = read_from_mastodon(path_mast)

In [ ]:
print(np.shape(raw_image))
scaling = [0.5, 0.5, 0.5]

In [ ]:
importlib.reload(main)
# GET THE REGISTERED MASTODON POSITIONS AT CURRENT TIMEPOIN
seeds_pos, seeds_array = main.get_seeds(lT, TP, VIEW, R_of_t, scaling, raw_image, trans_in_rev=True)

# PREPROCCESS IMAGE TO CREATE WATERSHED INPUT
im_for_ws = main.preprocess_im(raw_image)

In [ ]:
ws, all_seeds, missed_seeds, stat_table = main.ws_adaptive_mask(im_for_ws, seeds_array, min_int=np.percentile(im_for_ws.ravel(), 80), max_int=np.percentile(im_for_ws.ravel(), 99.9), 
                                                                percentage_list=[100, 99.99, 99.9, 99, 97, 95, 90, 85, 80, 75, 70, 65, 60, 55, 50, 45, 40, 35, 30, 25, 20, 15, 10])

In [ ]:
from beautifultable import BeautifulTable

table = BeautifulTable()
table.columns.header = ['Percentage', 'Percentile Value', 'Found Seeds', 'Missed Seeds']
for i in range(len(stat_table)):
    table.rows.insert(i, stat_table[i])
print(table)
print(f'Out of {len(all_seeds)} anotations in total, missed seeds: {stat_table[-1][-1]}({stat_table[-1][-1]/len(all_seeds)*100:.3g}%)')

In [ ]:
import napari

viewer = napari.Viewer()
viewer.add_image(im_for_ws)
viewer.add_labels(ws)
viewer.add_points(reg_pos_at_t, size = 5, face_color='#aaff00ff', border_color='black')
napari.run()